In [1]:
import copy
import os
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd

from Lib_classes.DataFrameFilesBuilderStack import DataFrameStackBuilder
from Lib_classes.FileArchitecture import PdPickleFileArchitecture
from Lib_classes.funcs import get_max_len_key, get_min_len_key, raise_none
from Lib_classes.Paths import Paths
from Source.Source_aggregate.SourceAggregator import SourceAggregator
from Source.Utils.Signature.BearingFaultSignature import BearingFaultSignature

In [3]:
pfa = PdPickleFileArchitecture.open(os.path.join(Paths.dir_dataframe_pickles, 'Bearing'))
df = pfa.get_file(0)

In [4]:
gen = np.random.default_rng(0)

In [5]:
n_chunks = 10

In [6]:
n_samples_in_chunk = len(df)//n_chunks
n_remains = len(df) - n_samples_in_chunk*n_chunks
n_remains

0

In [7]:
#becouse we have so many intersection of classes
def add_split_columns(df_):
    def reduce(x):
        if np.sum(x) == 0:
            return 0.0
        elif np.sum(x) == 1:
            return np.where(x > 1.0 - 10e-8)[0].item()+1
        elif np.sum(x) == 2:
            variants = np.where(x > 1.0 - 10e-8)[0]
            return float(variants[gen.integers(0, variants.size)])+1
            
    df_['group_index'] = df_[SourceAggregator.REAL_CLASS_ORDER].apply(reduce, axis = 1, raw=True) # type: ignore

    df_['ds_type'] = df_['original_path'].map(lambda x: Path(x).parts[len(Path(Paths.dir_source).absolute().parts)])

In [8]:
add_split_columns(df)
groups = df.groupby(['ds_type', 'group_index'])

In [9]:
map_groups_:dict[str,list] = {}
for key, group_ in groups.groups.items():
    map_groups_[f"{key[0]}_{key[1]}"] = list(group_)

In [10]:
remainders = []
for _ in range(n_remains):
    key = get_max_len_key(map_groups_)
    i = gen.choice(np.arange(len(map_groups_[key])))
    remainders.append(map_groups_[key].pop(i))

remainders

[]

In [11]:
map_chunks = defaultdict(list)
map_groups_remained = copy.deepcopy(map_groups_)
n_group_remains_determined = n_chunks


def split_for_chunks(group_:list, map_chunks_:defaultdict, n_group_remains):
    gen.shuffle(group_)
    need_size = len(group_)//n_chunks
    if need_size == 0:
        p = np.zeros((n_chunks,))
        for key, chunk in map_chunks_.items():
            p[key] = len(chunk)
        p = np.max(p)-p+1
        p = p/np.sum(p)
        
        if n_group_remains > n_group_remains_determined: 
            i_select = gen.choice(
                np.arange(n_chunks),
                size = (len(group_), ),
                p = p,
                replace = False
            )
            
            for chunk in i_select:
                map_chunks_[chunk].append(group_.pop(0))
        else:
            for _ in range(len(group_)):
                get_key = get_min_len_key(map_chunks_)
                map_chunks_[get_key].append(group_.pop())
            
        
    else:
        for chunk in range(n_chunks):
            if need_size != 0:
                current_group = group_[:need_size]
                if len(current_group) != need_size:
                    raise ValueError('len(current_group) != need_size')
                map_chunks_[chunk].extend(group_[:need_size])
                del group_[:need_size]
        
    if len(group_) != 0:
        split_for_chunks(group_, map_chunks_, n_group_remains)


c_groups = len(list(map_groups_.keys()))
for key, group in map_groups_remained.items():
    split_for_chunks(group, map_chunks, c_groups)
    c_groups-=1
    
map_chunks = dict(map_chunks)
        

In [12]:
map_groups_

{'CWRU_0': [507, 656],
 'CWRU_1': [23, 35, 196, 380, 462, 584, 619, 631, 722, 766, 940],
 'CWRU_2': [71, 76, 207, 216, 451, 464, 484, 517, 670, 805, 998],
 'CWRU_3': [31,
  36,
  60,
  75,
  81,
  164,
  181,
  219,
  365,
  576,
  585,
  660,
  745,
  751,
  764,
  841,
  850,
  856,
  925,
  945,
  985],
 'HUST_0': [253, 948],
 'HUST_2': [14, 544, 991],
 'HUST_3': [151, 303, 510, 526, 888],
 'KAt_0': [24,
  43,
  50,
  86,
  132,
  170,
  183,
  195,
  203,
  224,
  247,
  274,
  292,
  307,
  321,
  339,
  351,
  389,
  390,
  402,
  408,
  410,
  428,
  447,
  452,
  454,
  459,
  496,
  519,
  554,
  568,
  572,
  602,
  612,
  639,
  653,
  663,
  701,
  703,
  712,
  727,
  736,
  738,
  748,
  761,
  771,
  794,
  865,
  893,
  917,
  933,
  936,
  962,
  963],
 'KAt_1': [17,
  28,
  32,
  51,
  56,
  57,
  87,
  89,
  99,
  121,
  124,
  128,
  133,
  136,
  139,
  156,
  165,
  169,
  187,
  191,
  193,
  222,
  225,
  239,
  240,
  241,
  251,
  255,
  269,
  270,
  282,
  2

In [13]:
map_groups_remained

{'CWRU_0': [],
 'CWRU_1': [],
 'CWRU_2': [],
 'CWRU_3': [],
 'HUST_0': [],
 'HUST_2': [],
 'HUST_3': [],
 'KAt_0': [],
 'KAt_1': [],
 'KAt_3': [],
 'MFPT_0': [],
 'MFPT_1': [],
 'MFPT_3': [],
 'Multi-domain_0': [],
 'Multi-domain_1': [],
 'Multi-domain_2': [],
 'Multi-domain_3': [],
 'Multi-domain2_0': [],
 'Multi-domain2_1': [],
 'Multi-domain2_2': [],
 'Multi-domain2_3': [],
 'Multi-domain3_0': [],
 'Multi-domain3_1': [],
 'Multi-domain3_2': [],
 'Multi-domain3_3': [],
 'SCA_0': [],
 'SCA_1': [],
 'SCA_3': [],
 'Self-built dataset_1': [],
 'Self-built dataset_3': [],
 'SpectraQuest MFS dataset_1': [],
 'SpectraQuest MFS dataset_3': [],
 'XJTU-SY_0': [],
 'XJTU-SY_1': [],
 'XJTU-SY_3': []}

In [14]:
[f"{key}: {val}" for key, val in  map_chunks.items()]

['7: [507, 23, 216, 841, 60, 86, 50, 292, 274, 712, 898, 56, 461, 422, 883, 353, 438, 624, 156, 128, 813, 638, 341, 126, 272, 718, 583, 553, 114, 892, 678, 694, 691, 997, 145, 858, 249, 683, 278, 6, 279, 649, 802, 809, 885, 189, 620, 344, 608, 846, 543, 1, 798, 659, 362, 974, 245, 871, 396, 487, 995, 759, 460, 901, 820, 236, 538, 581, 662, 719, 836, 350, 123, 94, 244, 226, 325, 972, 867, 523, 102, 150, 539, 455, 116, 161, 385, 427, 929, 182, 409, 503, 83, 495, 987, 549, 866, 42, 511, 333]',
 '1: [656, 619, 805, 985, 585, 14, 510, 183, 771, 428, 454, 865, 852, 924, 469, 191, 937, 17, 338, 741, 187, 547, 903, 214, 262, 580, 935, 784, 661, 921, 790, 728, 53, 821, 906, 467, 144, 934, 482, 689, 364, 393, 340, 233, 525, 889, 839, 508, 811, 775, 108, 287, 405, 877, 690, 176, 375, 98, 725, 434, 223, 516, 753, 148, 629, 876, 915, 514, 328, 403, 91, 153, 79, 621, 288, 489, 817, 227, 147, 716, 932, 397, 965, 27, 651, 757, 373, 810, 597, 551, 212, 391, 93, 772, 989, 305, 522, 677, 563, 595]',
 '0:

In [15]:
[f"{key}: {len(val)}" for key, val in  map_chunks.items()]

['7: 100',
 '1: 100',
 '0: 100',
 '2: 100',
 '3: 100',
 '4: 100',
 '5: 100',
 '6: 100',
 '8: 100',
 '9: 100']

In [16]:
for i, key in zip(remainders, map_chunks.keys()):
    map_chunks[key].append(i)

In [17]:
[f"{key}: {len(val)}" for key, val in  map_chunks.items()]

['7: 100',
 '1: 100',
 '0: 100',
 '2: 100',
 '3: 100',
 '4: 100',
 '5: 100',
 '6: 100',
 '8: 100',
 '9: 100']

In [18]:
cum_df = None

for key, val in  map_chunks.items():
    new_df = pd.DataFrame(df.iloc[val].groupby(['ds_type', 'group_index']).size(), columns = ['size'])
    if cum_df is None:
        cum_df = new_df
    else:
        cum_df = pd.merge(cum_df, new_df, how = 'outer', on = ['ds_type', 'group_index'], suffixes = ('', f'_{key}'))
cum_df

size  size_1  size_0  size_2  size_3  \
ds_type                  group_index                                         
CWRU                     0             1.0     1.0     NaN     NaN     NaN   
                         1             1.0     1.0     1.0     1.0     1.0   
                         2             1.0     1.0     1.0     1.0     2.0   
                         3             2.0     2.0     3.0     2.0     2.0   
HUST                     0             NaN     NaN     NaN     1.0     NaN   
                         2             NaN     1.0     1.0     NaN     NaN   
                         3             NaN     1.0     NaN     1.0     NaN   
KAt                      0             5.0     5.0     5.0     5.0     6.0   
                         1            10.0     9.0    10.0    10.0     9.0   
                         3            12.0    11.0    12.0    12.0    12.0   
MFPT                     0             1.0     NaN     NaN     NaN     NaN   
                         1             NaN     NaN     NaN     NaN     NaN   
                         3             1.0     NaN     NaN     NaN     NaN   
Multi-domain             0             1.0     1.0     1.0     1.0     1.0   
                         1             1.0     NaN     1.0     NaN     1.0   
                         2             1.0     1.0     1.0     1.0     2.0   
                         3             1.0     1.0     1.0     1.0     1.0   
Multi-domain2            0             1.0     2.0     2.0     1.0     1.0   
                         1             1.0     1.0     1.0     1.0     1.0   
                         2             NaN     1.0     1.0     1.0     1.0   
                         3             1.0     1.0     1.0     1.0     1.0   
Multi-domain3            0             1.0     1.0     1.0     1.0     1.0   
                         1             1.0     1.0     1.0     1.0     1.0   
                         2             1.0     1.0     1.0     1.0     1.0   
                         3             1.0     1.0     1.0     1.0     1.0   
SCA                      0            36.0    37.0    35.0    36.0    35.0   
                         1             2.0     2.0     2.0     3.0     3.0   
                         3             NaN     NaN     NaN     NaN     NaN   
Self-built dataset       1             NaN     NaN     NaN     NaN     NaN   
                         3             1.0     NaN     NaN     NaN     NaN   
SpectraQuest MFS dataset 1             NaN     1.0     NaN     NaN     NaN   
                         3             NaN     NaN     1.0     NaN     NaN   
XJTU-SY                  0             5.0     4.0     4.0     5.0     5.0   
                         1             5.0     6.0     5.0     5.0     5.0   
                         3             6.0     6.0     7.0     7.0     7.0   

                                      size_4  size_5  size_6  size_8  size_9  
ds_type                  group_index                                          
CWRU                     0               NaN     NaN     NaN     NaN     NaN  
                         1               1.0     1.0     2.0     1.0     1.0  
                         2               1.0     1.0     1.0     1.0     1.0  
                         3               2.0     2.0     2.0     2.0     2.0  
HUST                     0               NaN     NaN     NaN     1.0     NaN  
                         2               1.0     NaN     NaN     NaN     NaN  
                         3               1.0     1.0     1.0     NaN     NaN  
KAt                      0               5.0     5.0     6.0     6.0     6.0  
                         1               9.0    10.0     9.0    10.0    10.0  
                         3              12.0    11.0    12.0    12.0    12.0  
MFPT                     0               NaN     NaN     NaN     NaN     NaN  
                         1               NaN     NaN     NaN     NaN     1.0  
                         3               NaN 

In [19]:
class EqualSizeSplitter:
    """
    Pure Fabrication responsible for splitting a dataset of indices into
    exactly N chunks of equal size, maintaining stratification.
    """
    def __init__(self,  rng: np.random.Generator, n_chunks: Optional[int] = None, size_chunk: Optional[int] = None, size: Optional[int] = None):
        self._rng = rng
        if size_chunk is not None and size is not None:
            self.n_chunks = size//size_chunk
            self.max_size_chunk = size_chunk
        elif n_chunks is not None:
            self.n_chunks = n_chunks
        self._n_group_remains_determined = self.n_chunks

    def split(self, groups: Dict[str, List[int]]) -> Dict[int, List[int]]:
        """
        Distributes indices from groups into chunks.
        
        1. Trims global excess to ensure total_size % n_chunks == 0.
        2. Distributes main body of groups evenly.
        3. Distributes remainders probabilistically to balance chunk sizes.
        """
        # Create a working copy to avoid side effects
        working_groups = {k: v.copy() for k, v in groups.items()}
        
        tail_idx = self._trim_global_excess(working_groups)
        
        chunks = defaultdict(list)
        
        # Sort keys to ensure deterministic processing order
        sorted_keys = sorted(working_groups.keys())
        c_working_groups = len(working_groups)
        for key in sorted_keys:
            indices = working_groups[key]
            self._rng.shuffle(indices)
            
            self._split_group(indices, chunks, c_working_groups)
            c_working_groups-=1
        
        chunks[max(chunks.keys())+1] = tail_idx
        for key, val in chunks.items():
            chunks[key] = self._rng.permutation(val).tolist()
        
        return dict(chunks)

    def _trim_global_excess(self, groups: Dict[str, List[int]]) -> list:
        """Removes indices from largest groups so total count fits n_chunks."""
        tail_idx = []
        total_samples = sum(len(v) for v in groups.values())
        if hasattr(self, 'max_size_chunk'):
            remainder = total_samples % self.max_size_chunk
        else:
            remainder = total_samples % self.n_chunks
        
        if remainder == 0:
            return []

        # Strategy: Remove from the largest groups to preserve rare classes
        for _ in range(remainder):
            largest_group_key = max(groups, key=lambda k: len(groups[k]))
            if groups[largest_group_key]:
                # Remove a random element
                idx_to_remove = self._rng.integers(0, len(groups[largest_group_key]))
                tail_idx.append(groups[largest_group_key].pop(idx_to_remove))
                
        return tail_idx

    def _split_group(self, indices: List[int], chunks: Dict[int, List[int]], n_group_remains:int) -> None:
        """Splits a single group's indices across chunks."""
        need_size = len(indices)//self.n_chunks
        if need_size == 0:
            self._split_underloaded_chunk(indices, chunks, n_group_remains)
        else:
            for chunk in range(self.n_chunks):
                if need_size != 0:
                    current_group = indices[:need_size]
                    if len(current_group) != need_size:
                        raise ValueError('len(current_group) != need_size')
                    chunks[chunk].extend(indices[:need_size])
                    del indices[:need_size]
        
        if len(indices) != 0:
            self._split_group(indices, chunks, n_group_remains)

    def _split_underloaded_chunk(self, indices: List[int], chunks: Dict[int, List[int]], n_group_remains:int ):
        """
        Selects a chunk index based on inverse probability of current size.
        Smaller chunks have higher probability of being selected.
        """
        p = np.zeros((self.n_chunks,))
        for key, chunk in chunks.items():
            p[key] = len(chunk)
        p = np.max(p)-p+1
        p = p/np.sum(p)
        
        if n_group_remains > self._n_group_remains_determined: 
            i_select = gen.choice(
                np.arange(self.n_chunks),
                size = (len(indices), ),
                p = p,
                replace = False
            )
            
            for chunk in i_select:
                chunks[chunk].append(indices.pop(0))
        else:
            for _ in range(len(indices)):
                get_key = min(chunks.keys(), key=lambda k: len(chunks[k]))
                chunks[get_key].append(indices.pop())
         


class BalancedChunkRunner:
    """
    Controller responsible for loading data and coordinating the split.
    """
    def __init__(self, 
                 pfa: PdPickleFileArchitecture,
                 spliter: EqualSizeSplitter,
                 signature_strategy: StratificationSignature,
                 rng: np.random.Generator
        ):
        self.pfa = pfa
        
        # Composition of helper classes
        self._signature_strategy = signature_strategy
        self._splitter = spliter
        self._rng = rng

    def generate_chunk_indices(self) -> Dict[int, List[int]]:
        """
        Loads a file, groups data by signature, and returns balanced chunk indices.
        """
        cum_groups = defaultdict(list)
        for i in self.pfa.iter_stack_index():
            df = self.pfa.get_file(i)
            groups = self._group_data(df, i)
            [cum_groups[k].extend(v) for k, v in groups.items()]
            
        chunk_map = self._splitter.split(groups)
        
        return chunk_map
    
    def provide_chunks(self, chunks:dict):
        for key, chunk in chunks.items():
            chunk_df = None
            for i in self.pfa.iter_stack_index():
                cur_chunks = list(map(lambda x: x[1], filter(lambda x: x[0] == i, chunk)))
                df = self.pfa.get_file(i).iloc[cur_chunks]
                chunk_df = pd.concat([chunk_df, df], ignore_index=True) if chunk_df is not None else df
            chunk_df = raise_none(chunk_df).sample(frac=1, random_state=self._rng.integers(0, np.iinfo(np.int32).max)).reset_index(drop=True)
            yield key, chunk_df
                    
            
            
            

    def _group_data(self, df: pd.DataFrame, c_file:int) -> Dict[str, List[int]]:
        """Groups DataFrame indices by their stratification signature."""
        groups = defaultdict(list)
        
        for idx, row in df.iterrows():
            signature = self._signature_strategy.get_signature(row)
            groups[signature].append((c_file, idx))
        return dict(groups)



NameError: name 'StratificationSignature' is not defined

In [ ]:
rng = np.random.default_rng(0)
bcr = BalancedChunkRunner(
    pfa,
    EqualSizeSplitter(rng, n_chunks = 10),
    BearingFaultSignature(SourceAggregator.REAL_CLASS_ORDER, rng),
    rng
)
chunks = bcr.generate_chunk_indices()

In [ ]:
len(pfa.rows())

8843

In [ ]:
[f"{key}: {len(val)}" for key, val in  chunks.items()]

['0: 884',
 '1: 884',
 '2: 884',
 '3: 884',
 '4: 884',
 '5: 884',
 '6: 884',
 '7: 884',
 '8: 884',
 '9: 884',
 '10: 3']

In [ ]:
cum_df = None

for key, val in  chunks.items():
    # print(key, val)
    new_df = pd.DataFrame(df.iloc[list(map(lambda x: x[1], val))].groupby(['ds_type', 'group_index']).size(), columns = ['size'])
    if cum_df is None:
        cum_df = new_df
    else:
        cum_df = pd.merge(cum_df, new_df, how = 'outer', on = ['ds_type', 'group_index'], suffixes = ('', f'_{key}'))
cum_df

size  size_1  size_2  size_3  size_4  \
ds_type                  group_index                                          
CWRU                     0              1.0     1.0     1.0     1.0     1.0   
                         1             11.0    10.0    10.0    11.0    11.0   
                         2             10.0    10.0    10.0    10.0    10.0   
                         3             20.0    20.0    21.0    20.0    20.0   
HUST                     0              1.0     1.0     2.0     1.0     1.0   
                         1              3.0     3.0     2.0     3.0     3.0   
                         2              3.0     3.0     3.0     3.0     2.0   
                         3              3.0     4.0     3.0     3.0     3.0   
JNU                      0              NaN     1.0     NaN     NaN     1.0   
                         1              1.0     NaN     NaN     NaN     NaN   
                         2              NaN     NaN     1.0     1.0     NaN   
                         3              NaN     1.0     NaN     NaN     1.0   
MFPT                     0              NaN     NaN     1.0     NaN     NaN   
                         1              1.0     NaN     NaN     NaN     1.0   
                         3              1.0     1.0     1.0     1.0     1.0   
Multi-domain             0             10.0     9.0    10.0    10.0    10.0   
                         1             10.0    10.0    10.0     9.0    10.0   
                         2             10.0    10.0    10.0     9.0    10.0   
                         3             10.0    10.0     9.0    10.0     9.0   
Multi-domain2            0             10.0     9.0    10.0    10.0    10.0   
                         1             10.0    10.0     9.0     9.0     9.0   
                         2              9.0     9.0    10.0     9.0    10.0   
                         3              9.0     9.0     9.0    10.0    10.0   
Multi-domain3            0              9.0     9.0    10.0    10.0    10.0   
                         1             10.0    10.0     9.0    10.0     9.0   
                         2              9.0    10.0    10.0     9.0     9.0   
                         3              9.0     9.0     9.0    10.0     9.0   
SCA                      0            482.0   481.0   481.0   482.0   481.0   
                         1             34.0    35.0    35.0    35.0    35.0   
                         2              2.0     2.0     2.0     2.0     2.0   
                         3             25.0    25.0    25.0    24.0    25.0   
Self-built dataset       1              1.0     1.0     1.0     1.0     1.0   
                         2              1.0     NaN     NaN     NaN     1.0   
                         3              NaN     2.0     1.0     2.0     NaN   
SpectraQuest MFS dataset 1              1.0     1.0     1.0     1.0     1.0   
                         2              1.0     1.0     NaN     NaN     NaN   
                         3              1.0     NaN     1.0     1.0     1.0   
XJTU-SY                  0             40.0    41.0    41.0    41.0    41.0   
                         1             52.0    52.0    52.0    51.0    51.0   
                         3             63.0    63.0    63.0    64.0    64.0   
fm6xzxnf36-2             0              1.0     1.0     NaN     NaN     NaN   
                         1              5.0     5.0     6.0     6.0     6.0   
                         3              5.0     5.0     5.0     5.0     5.0   

                                      size_5  size_6  size_7  size_8  size_9  \
ds_type                  group_index                                           
CWRU                     0               1.0     1.0     1.0     1.0     1.0   
                         1              10.0    10.0    10.0    10.0    10.0   
                         2              10.0    10.0    10.0    10.0    10.0   
                         3              20.0    20.0    21.0    20.0    21.0   
HUS

In [ ]:


FOLDER_NAME = 'balancer_chunks'
path = os.path.join(Paths.dir_dataframe_pickles, 'Bearing', FOLDER_NAME)

In [ ]:
pmf = DataFrameStackBuilder(
    list(pfa.get_columns()), 
    PdPickleFileArchitecture(
        path,
        max_stack_size=len(next(iter(chunks.values())))
    )
)

for k_, df in bcr.provide_chunks(chunks):
    pmf.add_df(df)

pmf.save_tail()

In [ ]:
pfa_chunks = PdPickleFileArchitecture.open(path)

In [ ]:
cum_df = None

for i in pfa_chunks.iter_stack_index():
    df = pfa_chunks.get_file(i)
    add_split_columns(df)
    new_df = pd.DataFrame(df.groupby(['ds_type', 'group_index']).size(), columns = ['size'])
    if cum_df is None:
        cum_df = new_df
    else:
        print(i)
        cum_df = pd.merge(cum_df, new_df, how = 'outer', on = ['ds_type', 'group_index'], suffixes = ('', f'_{i}'))
cum_df

1
2
3
4
5
6
7
8
9
10


size  size_1  size_2  size_3  size_4  \
ds_type                  group_index                                          
CWRU                     0.0            1.0     1.0     1.0     1.0     1.0   
                         1.0           10.0    10.0    10.0    10.0    10.0   
                         2.0           10.0    10.0    10.0    10.0    10.0   
                         3.0           20.0    20.0    20.0    20.0    21.0   
HUST                     0.0            1.0     2.0     2.0     1.0     1.0   
                         1.0            4.0     2.0     3.0     3.0     3.0   
                         2.0            2.0     1.0     4.0     2.0     3.0   
                         3.0            3.0     6.0     1.0     3.0     3.0   
JNU                      0.0            NaN     NaN     1.0     NaN     NaN   
                         1.0            NaN     NaN     NaN     NaN     1.0   
                         2.0            1.0     NaN     1.0     1.0     NaN   
                         3.0            NaN     NaN     NaN     1.0     NaN   
MFPT                     0.0            NaN     1.0     NaN     1.0     NaN   
                         1.0            1.0     1.0     1.0     NaN     1.0   
                         3.0            1.0     1.0     1.0     1.0     1.0   
Multi-domain             0.0           10.0    10.0     9.0    10.0     9.0   
                         1.0           10.0     9.0    10.0     9.0    10.0   
                         2.0           10.0     9.0     9.0    10.0     9.0   
                         3.0            9.0     9.0     9.0    10.0    10.0   
Multi-domain2            0.0            9.0     9.0    10.0    10.0    10.0   
                         1.0           10.0    10.0    10.0     9.0     9.0   
                         2.0            9.0    10.0     9.0    10.0    10.0   
                         3.0           10.0    10.0    10.0    10.0     9.0   
Multi-domain3            0.0            9.0    10.0    10.0    10.0     9.0   
                         1.0           10.0     9.0    10.0     9.0    10.0   
                         2.0           10.0     9.0     9.0    10.0    10.0   
                         3.0            9.0    10.0     9.0     9.0    10.0   
SCA                      0.0          482.0   482.0   482.0   481.0   481.0   
                         1.0           35.0    35.0    35.0    35.0    34.0   
                         2.0            2.0     2.0     2.0     2.0     3.0   
                         3.0           24.0    25.0    25.0    25.0    25.0   
Self-built dataset       1.0            1.0     1.0     1.0     1.0     1.0   
                         2.0            1.0     NaN     1.0     NaN     NaN   
                         3.0            1.0     1.0     NaN     1.0     1.0   
SpectraQuest MFS dataset 1.0            1.0     1.0     1.0     1.0     1.0   
                         2.0            1.0     1.0     NaN     NaN     NaN   
                         3.0            1.0     NaN     1.0     1.0     1.0   
XJTU-SY                  0.0           40.0    41.0    41.0    41.0    41.0   
                         1.0           53.0    52.0    51.0    53.0    52.0   
                         3.0           62.0    63.0    64.0    62.0    63.0   
fm6xzxnf36-2             0.0            1.0     1.0     NaN     NaN     NaN   
                         1.0            5.0     5.0     6.0     6.0     6.0   
                         3.0            5.0     5.0     5.0     5.0     5.0   

                                      size_5  size_6  size_7  size_8  size_9  \
ds_type                  group_index                                           
CWRU                     0.0             1.0     1.0     1.0     1.0     1.0   
                         1.0            11.0    10.0    11.0    10.0    11.0   
                         2.0            10.0    10.0    10.0    10.0    10.0   
                         3.0            21.0    21.0    20.0    20.0    20.0   
HUS

In [ ]:
SourceAggregator.split_chunks(10)

Using cuda:0 device


11it [01:34,  8.62s/it]
